<a href="https://colab.research.google.com/github/detektor777/colab_list_video/blob/main/osediff_video_upscale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
show_log = False #@param {type:"boolean"}

import os, shutil, re, subprocess as sp
from IPython.utils import io
import contextlib

@contextlib.contextmanager
def optional_capture(capture):
    if capture:
        with io.capture_output() as captured:
            yield captured
    else:
        yield None

with optional_capture(not show_log):
    if os.path.isdir('/content/OSEDiff'):
        shutil.rmtree('/content/OSEDiff')

    os.system('git clone -q --depth 1 https://github.com/cswry/OSEDiff.git')
    os.chdir('OSEDiff')

    # The repository's requirements.txt pins versions of some old packages (e.g.,
    # PyYAML==5.4.1), which lack precompiled wheels for the current Python/setuptools
    # in Colab — pip tries to build them from source and fails with "Getting requirements
    # to build wheel did not run successfully" EVEN BEFORE it reaches diffusers/
    # transformers/peft/accelerate/basicsr. Fix: remove torch/torchvision/
    # torchaudio/xformers (we use what is already in Colab) and UNPIN the strict
    # versions for all other packages, leaving only names — pip will pick the
    # actual version with a precompiled wheel itself.
    with open('requirements.txt') as f:
        raw_lines = f.readlines()

    cleaned = []
    for line in raw_lines:
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('--extra-index-url'):
            continue
        if re.match(r'^(torch|torchvision|torchaudio|xformers)([=<>~!]|$)', line):
            continue
        pkg = re.split(r'[=<>~!\[]', line)[0].strip()
        if pkg:
            cleaned.append(pkg)

    with open('requirements_colab.txt', 'w') as f:
        f.write('\n'.join(cleaned) + '\n')

    print('requirements_colab.txt:')
    print(open('requirements_colab.txt').read())

    # --- pip install with exit code check ----------------------------------------
    install_result = os.system('pip install -q -r requirements_colab.txt')
    if install_result != 0:
        print('⚠️ pip install returned an error, trying to install packages one by one '
              '(so that one problematic package does not block the rest)...')
        with open('requirements_colab.txt') as f:
            pkgs = [p.strip() for p in f if p.strip()]
        for pkg in pkgs:
            r = os.system(f'pip install -q "{pkg}"')
            if r != 0:
                print(f'   ⚠️ failed to install: {pkg} (skipping, see output above)')

    # accelerate is not explicitly listed in requirements.txt (pulled as a dependency
    # by diffusers/peft), but is explicitly needed for inference — installing separately just in case.
    os.system('pip install -q accelerate')
    os.system('pip install -q huggingface_hub')

    # Extra packages needed for the VIDEO pipeline (frame extraction / video assembly),
    # not required by the original single-image OSEDiff repo.
    os.system('pip install -q opencv-python-headless ffmpeg-python imageio tqdm ipywidgets')

    # --- Patch torchao: version incompatibility with recent peft -----------------
    # Colab defaults to torchao 0.10.0, and the current peft installed alongside
    # requirements_colab.txt calls unet.add_adapter(...) (see osediff.py::load_ckpt)
    # going into peft/tuners/lora/torchao.py::dispatch_torchao and calling
    # is_torchao_available(), which explicitly requires torchao >= 0.16.0,
    # otherwise it throws an ImportError before the LoRA adapter is even applied.
    # Upgrading torchao separately (after the main installation to avoid pulling
    # torch version conflicts during requirements_colab.txt dependency resolution).
    torchao_upgrade_result = os.system('pip install -q -U "torchao>=0.16.0"')
    if torchao_upgrade_result != 0:
        print('⚠️ Failed to upgrade torchao to version >=0.16.0 — LoRA adapter '
              '(unet.add_adapter) at the "Run" step might fail with ImportError.')
    else:
        print('torchao upgraded to version >=0.16.0 (required for peft/dispatch_torchao).')

    # --- Patch basicsr: incompatibility with recent torchvision ------------------
    # basicsr/data/degradations.py does `from torchvision.transforms.functional_tensor
    # import rgb_to_grayscale` — this module was removed from torchvision starting version
    # 0.17 (the function moved to torchvision.transforms.functional), causing a simple
    # `import basicsr` to crash with ModuleNotFoundError before any use.
    # Patching the import to the actual path in all installed files where it occurs.
    find_result = sp.run(
        ['bash', '-c',
         "python -c \"import basicsr, os; print(os.path.dirname(basicsr.__file__))\" 2>/dev/null"],
        capture_output=True, text=True
    )
    basicsr_dir = find_result.stdout.strip()
    if not basicsr_dir:
        show_result = sp.run(['pip', 'show', 'basicsr'], capture_output=True, text=True).stdout
        for line in show_result.splitlines():
            if line.startswith('Location:'):
                site_packages = line.split(':', 1)[1].strip()
                basicsr_dir = os.path.join(site_packages, 'basicsr')
                break

    if basicsr_dir and os.path.isdir(basicsr_dir):
        patched_files = sp.run(
            ['grep', '-rl', 'functional_tensor', basicsr_dir],
            capture_output=True, text=True
        ).stdout.split()
        for fpath in patched_files:
            os.system(
                f"sed -i 's/from torchvision\\.transforms\\.functional_tensor import rgb_to_grayscale/"
                f"from torchvision.transforms.functional import rgb_to_grayscale/' {fpath}"
            )
        print('basicsr: patched files:', patched_files or '(none required)')
    else:
        print('⚠️ Failed to find basicsr directory for patching — please check the installation manually.')

    # --- Patch RAM: hardcoded local path to bert-base-uncased --------------------
    # ram/models/utils.py::init_tokenizer() in the original repo references
    # the author's absolute local path (something like
    # '/home/notebook/data/group/LowLevelLLM/LLM/bert-base-uncased'), which does not
    # exist in Colab or any other machine. This causes test_osediff.py to crash with
    # huggingface_hub.errors.HFValidationError / "Can't load tokenizer"
    # during RAM model initialization (see issue #7 in cswry/OSEDiff).
    # Replacing the hardcoded path with 'bert-base-uncased' so the tokenizer
    # is downloaded from Hugging Face Hub as intended.
    ram_utils_candidates = sp.run(
        ['grep', '-rl', 'bert-base-uncased', os.getcwd()],
        capture_output=True, text=True
    ).stdout.split()

    bert_path_pattern = re.compile(r"""(['"])(?:/[^'"]*)?bert-base-uncased\1""")
    ram_patched_files = []
    for fpath in ram_utils_candidates:
        with open(fpath, encoding='utf-8') as f:
            src = f.read()
        new_src = bert_path_pattern.sub(lambda m: f"{m.group(1)}bert-base-uncased{m.group(1)}", src)
        if new_src != src:
            with open(fpath, 'w', encoding='utf-8') as f:
                f.write(new_src)
            ram_patched_files.append(fpath)

    if ram_patched_files:
        print('RAM tokenizer: hardcoded path to bert-base-uncased replaced with HF repo id in files:', ram_patched_files)
    else:
        print('RAM tokenizer patch: no changes required (path is already correct or file not found).')

    # --- Patch RAM: unreliable additional_special_tokens_ids property ------------
    # ram/models/utils.py::init_tokenizer() does
    # `tokenizer.enc_token_id = tokenizer.additional_special_tokens_ids[0]`.
    # In current transformers versions, this property under certain conditions
    # throws an internal AttributeError, which Python interprets as "attribute missing",
    # and BertTokenizer gives a misleading message "BertTokenizer has no attribute
    # additional_special_tokens_ids" before the RAM model even loads.
    # The special token '[ENC]' is already added via add_special_tokens by this point,
    # so we just get its ID directly via convert_tokens_to_ids — this is more reliable
    # and independent of the transformers version.
    # Also removing local_files_only=True from from_pretrained so the tokenizer
    # can download itself from Hugging Face Hub if it's missing in the local cache.
    utils_path = os.path.join(os.getcwd(), 'ram', 'models', 'utils.py')
    if os.path.exists(utils_path):
        with open(utils_path, encoding='utf-8') as f:
            utils_src = f.read()

        old_enc_line = "tokenizer.enc_token_id = tokenizer.additional_special_tokens_ids[0]"
        new_enc_line = "tokenizer.enc_token_id = tokenizer.convert_tokens_to_ids('[ENC]')"
        replaced_enc = old_enc_line in utils_src
        if replaced_enc:
            utils_src = utils_src.replace(old_enc_line, new_enc_line)

        old_local_only = ", local_files_only=True)"
        replaced_local_only = old_local_only in utils_src
        if replaced_local_only:
            utils_src = utils_src.replace(old_local_only, ")")

        if replaced_enc or replaced_local_only:
            with open(utils_path, 'w', encoding='utf-8') as f:
                f.write(utils_src)
            print(f'ram/models/utils.py: init_tokenizer() patched '
                  f'(enc_token_id={replaced_enc}, local_files_only removed={replaced_local_only}).')
        else:
            print('⚠️ Did not find expected lines in ram/models/utils.py::init_tokenizer — perhaps the file changed, patch skipped.')
    else:
        print('⚠️ File ram/models/utils.py not found — skipping init_tokenizer patch.')

    # --- Patch RAM/BERT: transformers removed/moved several names, including -----
    # --- get_head_mask method (removed from PreTrainedModel/ModuleUtilsMixin) ----
    # ram/models/bert.py AND ram/models/bert_lora.py (adapted BERT code from
    # an old transformers version, two almost identical files) make the same
    # import `from transformers.modeling_utils import (PreTrainedModel,
    # apply_chunking_to_forward, find_pruneable_heads_and_indices, prune_linear_layer)`.
    # In current transformers, apply_chunking_to_forward and prune_linear_layer
    # moved to transformers.pytorch_utils, and find_pruneable_heads_and_indices
    # was removed entirely (used only for manual BERT attention head pruning,
    # which RAM inference in OSEDiff does not do). In addition, the get_head_mask
    # method (and its helper _convert_head_mask_to_5d) was removed from PreTrainedModel,
    # which BertModel.forward() calls directly as self.get_head_mask(...) —
    # causing AttributeError: 'BertModel' object has no attribute 'get_head_mask'.
    # Patching the import to a fault-tolerant version in ALL repo files where it occurs,
    # and simultaneously monkeypatching both methods back onto the PreTrainedModel
    # class if they are missing — this fixes it once for all models using this class.
    old_import = (
        "from transformers.modeling_utils import (\n"
        "    PreTrainedModel,\n"
        "    apply_chunking_to_forward,\n"
        "    find_pruneable_heads_and_indices,\n"
        "    prune_linear_layer,\n"
        ")"
    )
    new_import = (
        "import transformers.modeling_utils as _mod_transformers_modeling_utils\n"
        "try:\n"
        "    import transformers.pytorch_utils as _mod_transformers_pytorch_utils\n"
        "except ModuleNotFoundError:\n"
        "    _mod_transformers_pytorch_utils = None\n"
        "def _resolve_transformers_symbol(name):\n"
        "    for _mod in (_mod_transformers_modeling_utils, _mod_transformers_pytorch_utils):\n"
        "        if _mod is not None and hasattr(_mod, name):\n"
        "            return getattr(_mod, name)\n"
        "    return type(name, (object,), {})\n"
        "PreTrainedModel = _resolve_transformers_symbol('PreTrainedModel')\n"
        "apply_chunking_to_forward = _resolve_transformers_symbol('apply_chunking_to_forward')\n"
        "find_pruneable_heads_and_indices = _resolve_transformers_symbol('find_pruneable_heads_and_indices')\n"
        "prune_linear_layer = _resolve_transformers_symbol('prune_linear_layer')\n"
        "if not hasattr(PreTrainedModel, 'get_head_mask'):\n"
        "    def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):\n"
        "        if head_mask.dim() == 1:\n"
        "            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)\n"
        "            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)\n"
        "        elif head_mask.dim() == 2:\n"
        "            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)\n"
        "        head_mask = head_mask.to(dtype=self.dtype)\n"
        "        return head_mask\n"
        "    def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):\n"
        "        if head_mask is not None:\n"
        "            head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)\n"
        "            if is_attention_chunked is True:\n"
        "                head_mask = head_mask.unsqueeze(-1)\n"
        "        else:\n"
        "            head_mask = [None] * num_hidden_layers\n"
        "        return head_mask\n"
        "    PreTrainedModel._convert_head_mask_to_5d = _convert_head_mask_to_5d\n"
        "    PreTrainedModel.get_head_mask = _get_head_mask"
    )

    bert_patched_files = []
    for root, dirs, files in os.walk(os.getcwd()):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                file_src = f.read()
            if old_import in file_src:
                file_src = file_src.replace(old_import, new_import)
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(file_src)
                bert_patched_files.append(os.path.relpath(fpath, os.getcwd()))

    if bert_patched_files:
        print('Import from transformers.modeling_utils made fault-tolerant (+ get_head_mask) in files:', bert_patched_files)
    else:
        print('⚠️ Did not find the expected transformers.modeling_utils import block in any file — perhaps the repo structure changed.')

    # --- Patch RAM/BERT: init_weights() is deprecated, post_init() is required -
    # ram/models/bert.py and ram/models/bert_lora.py at the end of __init__
    # for each model class (BertModel, BertLMHeadModel, etc.) call self.init_weights() —
    # this is how it was done in older versions of transformers. In current versions,
    # init_weights() expects self.all_tied_weights_keys to be precalculated, but
    # it is only calculated by post_init() (which itself calls init_weights() at the end).
    # Calling init_weights() directly bypassing post_init() causes
    # AttributeError: 'BertModel' object has no attribute 'all_tied_weights_keys'
    # inside tie_weights(). Changing self.init_weights() to self.post_init()
    # in all repo files where such a call occurs.
    init_weights_patched_files = []
    for root, dirs, files in os.walk(os.getcwd()):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                file_src = f.read()
            new_file_src = re.sub(
                r'(?<!def )(?<!\.)\bself\.init_weights\(\)',
                'self.post_init()',
                file_src,
            )
            if new_file_src != file_src:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(new_file_src)
                init_weights_patched_files.append(os.path.relpath(fpath, os.getcwd()))

    if init_weights_patched_files:
        print('self.init_weights() replaced with self.post_init() in files:', init_weights_patched_files)
    else:
        print('⚠️ Did not find self.init_weights() calls in any file — perhaps the repo structure changed.')

    # --- Patch osediff.py: scheduler.timesteps might not match hardcoded 999 ---
    # In all classes (OSEDiff_test, etc.) the code does:
    #   self.noise_scheduler.set_timesteps(1, device="cuda")
    #   ...
    #   self.timesteps = torch.tensor([999], device="cuda").long()
    # and then calls self.noise_scheduler.step(model_pred, self.timesteps, ...),
    # which internally (DDPMScheduler.previous_timestep) searches for the index
    # of the value 999 in ITS OWN internal list self.noise_scheduler.timesteps,
    # calculated via set_timesteps(1, ...). In current versions of diffusers for
    # num_inference_steps=1, this list depending on timestep_spacing in the scheduler config
    # is not necessarily equal to [999] (it could be [0], for example) — then 999
    # is not found and it crashes with IndexError: index 0 is out of bounds for dimension
    # 0 with size 0. Fixed by forcibly overwriting self.noise_scheduler.timesteps
    # to [999] right after set_timesteps(1, ...), so it matches the model's hardcode.
    scheduler_timesteps_patched_files = []
    for root, dirs, files in os.walk(os.getcwd()):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                file_src = f.read()

            old_set_timesteps_line = 'self.noise_scheduler.set_timesteps(1, device="cuda")'
            new_set_timesteps_block = (
                'self.noise_scheduler.set_timesteps(1, device="cuda")\n'
                '        self.noise_scheduler.timesteps = torch.tensor([999], device="cuda").long()'
            )
            if old_set_timesteps_line in file_src:
                file_src = file_src.replace(old_set_timesteps_line, new_set_timesteps_block)
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(file_src)
                scheduler_timesteps_patched_files.append(os.path.relpath(fpath, os.getcwd()))

    if scheduler_timesteps_patched_files:
        print('noise_scheduler.timesteps forcibly set to [999] in files:', scheduler_timesteps_patched_files)
    else:
        print('⚠️ Did not find set_timesteps(1, device="cuda") calls in any file — perhaps the repo structure changed.')

    # --- Patch UNet2DConditionModel: add_adapter moved to PeftAdapterMixin -----
    # Local copy of models/unet_2d_condition.py is a diffusers fork for an older version
    # of the library, when LoRA/PEFT methods (add_adapter/delete_adapter, etc.) were
    # part of UNet2DConditionLoadersMixin. In current diffusers, they were moved to
    # a separate PeftAdapterMixin, and the base classes here are declared without it —
    # causing self.unet.add_adapter(...) to crash with AttributeError. Adding
    # PeftAdapterMixin to the base classes list (via getattr with a fallback stub
    # in case it is also missing in some diffusers version).
    unet_cond_path = os.path.join(os.getcwd(), 'models', 'unet_2d_condition.py')
    if os.path.exists(unet_cond_path):
        with open(unet_cond_path, encoding='utf-8') as f:
            unet_src = f.read()

        old_class_decl = 'class UNet2DConditionModel(ModelMixin, ConfigMixin, UNet2DConditionLoadersMixin):'
        if old_class_decl in unet_src:
            peft_mixin_shim = (
                "import diffusers.loaders as _mod_diffusers_loaders_for_peft\n"
                "PeftAdapterMixin = getattr(_mod_diffusers_loaders_for_peft, 'PeftAdapterMixin', "
                "type('PeftAdapterMixin', (object,), {}))\n\n\n"
            )
            new_class_decl = (
                'class UNet2DConditionModel(ModelMixin, ConfigMixin, UNet2DConditionLoadersMixin, PeftAdapterMixin):'
            )
            unet_src = unet_src.replace(old_class_decl, peft_mixin_shim + new_class_decl)
            with open(unet_cond_path, 'w', encoding='utf-8') as f:
                f.write(unet_src)
            print('models/unet_2d_condition.py: added PeftAdapterMixin to UNet2DConditionModel base classes (needed for add_adapter).')
        else:
            print('⚠️ Did not find expected UNet2DConditionModel class declaration — perhaps the file changed, patch skipped.')
    else:
        print('⚠️ File models/unet_2d_condition.py not found — skipping PeftAdapterMixin patch.')

    # --- Patch AutoencoderKL: add_adapter also moved to PeftAdapterMixin -------
    # Similar to UNet2DConditionModel, the local copy of models/autoencoder_kl.py is
    # a diffusers fork for an older version, where LoRA/PEFT methods (add_adapter, etc.)
    # were available without a separate mixin. In current diffusers, they were moved
    # to PeftAdapterMixin, and the AutoencoderKL base classes here are declared without it —
    # causing self.vae.add_adapter(...) in osediff.py::load_ckpt to crash with
    # AttributeError: 'AutoencoderKL' object has no attribute 'add_adapter'.
    vae_path = os.path.join(os.getcwd(), 'models', 'autoencoder_kl.py')
    if os.path.exists(vae_path):
        with open(vae_path, encoding='utf-8') as f:
            vae_src = f.read()

        old_vae_class_decl = 'class AutoencoderKL(ModelMixin, ConfigMixin, FromOriginalVAEMixin):'
        if old_vae_class_decl in vae_src:
            vae_peft_mixin_shim = (
                "import diffusers.loaders as _mod_diffusers_loaders_for_peft_vae\n"
                "PeftAdapterMixin = getattr(_mod_diffusers_loaders_for_peft_vae, 'PeftAdapterMixin', "
                "type('PeftAdapterMixin', (object,), {}))\n\n\n"
            )
            new_vae_class_decl = (
                'class AutoencoderKL(ModelMixin, ConfigMixin, FromOriginalVAEMixin, PeftAdapterMixin):'
            )
            vae_src = vae_src.replace(old_vae_class_decl, vae_peft_mixin_shim + new_vae_class_decl)
            with open(vae_path, 'w', encoding='utf-8') as f:
                f.write(vae_src)
            print('models/autoencoder_kl.py: added PeftAdapterMixin to AutoencoderKL base classes (needed for add_adapter).')
        else:
            print('⚠️ Did not find expected AutoencoderKL class declaration — perhaps the file changed, patch skipped.')
    else:
        print('⚠️ File models/autoencoder_kl.py not found — skipping PeftAdapterMixin patch.')

    # --- Universal patch: OSEDiff was written for an older version of diffusers -
    # In the installed Colab diffusers version, some classes (loading mixins,
    # individual embedding modules like PositionNet, etc.) have been renamed or
    # removed, and some submodule paths (e.g. diffusers.models.unet_2d_blocks)
    # ENTIRELY moved to the diffusers.models.unets.* subpackage. Instead of
    # fixing one ImportError after another precisely, we scan ALL .py files
    # in the repo and rewrite EVERY import like `from diffusers... import (A, B, C)`
    # (including multi-line ones, in parentheses) to a safe version: first try
    # to import the module by its original path, and if it no longer exists
    # (ModuleNotFoundError) — try the current diffusers.models.unets.<rest> path.
    # Then, for each name, use getattr(module, 'Name', fallback_stub). If the
    # symbol actually exists in the installed diffusers version — it is used
    # (no functionality loss), if removed/renamed — an empty stub class is
    # substituted instead of crashing on import.
    import_pattern = re.compile(
        r'^(?P<indent>[ \t]*)from (?P<module>diffusers[\w.]*) import '
        r'(?P<body>\([^()]*\)|[^\n]+)$',
        re.MULTILINE
    )

    def _parse_names(body):
        body = body.strip()
        if body.startswith('(') and body.endswith(')'):
            body = body[1:-1]
        names = []
        for part in body.split(','):
            part = part.split('#')[0].strip()
            if not part:
                continue
            if ' as ' in part:
                orig, alias = [p.strip() for p in part.split(' as ', 1)]
                names.append((orig, alias))
            else:
                names.append((part, part))
        return names

    repo_root = os.getcwd()
    patched_summary = []
    for root, dirs, files in os.walk(repo_root):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                src = f.read()

            matches = list(import_pattern.finditer(src))
            if not matches:
                continue

            new_src = src
            offset = 0
            touched_names = []
            for m in matches:
                module = m.group('module')
                names = _parse_names(m.group('body'))
                if not names:
                    continue
                mod_alias = '_mod_' + re.sub(r'\W', '_', module)

                fallback_module = None
                if module.startswith('diffusers.models.') and '.unets.' not in module:
                    fallback_module = module.replace('diffusers.models.', 'diffusers.models.unets.', 1)

                if fallback_module:
                    lines = [
                        f'{m.group("indent")}try:',
                        f'{m.group("indent")}    import {module} as {mod_alias}',
                        f'{m.group("indent")}except ModuleNotFoundError:',
                        f'{m.group("indent")}    import {fallback_module} as {mod_alias}',
                    ]
                else:
                    lines = [f'{m.group("indent")}import {module} as {mod_alias}']

                for orig, alias in names:
                    lines.append(
                        f"{m.group('indent')}{alias} = getattr({mod_alias}, '{orig}', "
                        f"type('{orig}', (object,), {{}}))"
                    )
                    touched_names.append(orig)
                replacement = '\n'.join(lines)

                start, end = m.start() + offset, m.end() + offset
                new_src = new_src[:start] + replacement + new_src[end:]
                offset += len(replacement) - (m.end() - m.start())

            if new_src != src:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(new_src)
                rel_path = os.path.relpath(fpath, repo_root)
                patched_summary.append(f'{rel_path}: {touched_names}')

    if patched_summary:
        print('diffusers imports patched to a safe (getattr) version:')
        for line in patched_summary:
            print('  -', line)
    else:
        print('diffusers imports patch: no changes required.')

    # --- Check that critical packages can actually be imported -----------------
    required_modules = ['diffusers', 'transformers', 'peft', 'accelerate', 'basicsr', 'einops']
    missing = []
    for module_name in required_modules:
        try:
            __import__(module_name)
        except ImportError as e:
            missing.append(f'{module_name} ({e})')

    if missing:
        raise RuntimeError(
            f'Critical modules failed to install/import: {missing}. '
            f'If "show_log" is off, please enable it, re-run the cell, and check the output for errors.'
        )
    else:
        print('✅ All critical packages are installed and importable:', required_modules)

    # --- Final check: the osediff package itself and both RAM versions must be importable ---
    # Checking not only osediff, but also ram.models.ram (uses bert.py) and
    # ram.models.ram_lora (uses bert_lora.py) — this is where transformers imports
    # used to crash, while a simple `import osediff` wouldn't touch them.
    check = sp.run(
        ['python', '-c',
         'import sys; sys.path.insert(0, "."); import osediff; import ram.models.ram; import ram.models.ram_lora; print("osediff OK")'],
        capture_output=True, text=True
    )
    print(check.stdout.strip())
    if check.returncode != 0:
        print('⚠️ The osediff/ram module still cannot be imported. Full traceback:')
        print(check.stderr)
        raise RuntimeError(
            'import osediff/ram failed with an error — see the traceback above '
            '(enable "show_log" if it is disabled) and provide its text.'
        )

    import torch
    print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

    # --- Weights -----------------------------------------------------------------
    from huggingface_hub import hf_hub_download, snapshot_download

    os.makedirs('preset/models', exist_ok=True)

    def file_ok(path, min_bytes=1024):
        return os.path.exists(path) and os.path.getsize(path) >= min_bytes

    sd21_dir = 'preset/models/stable-diffusion-2-1-base'
    if not os.path.isdir(sd21_dir) or not os.path.exists(os.path.join(sd21_dir, 'model_index.json')):
        print('Downloading SD21 Base (Manojb/stable-diffusion-2-1-base)...')
        snapshot_download(
            repo_id='Manojb/stable-diffusion-2-1-base',
            local_dir=sd21_dir,
            allow_patterns=[
                'tokenizer/*', 'text_encoder/*', 'vae/*', 'unet/*', 'scheduler/*',
                'model_index.json',
            ],
        )
        print('Done:', sd21_dir)
    else:
        print('SD21 Base is already present, skipping download:', sd21_dir)

    ram_path = 'preset/models/ram_swin_large_14m.pth'
    if not file_ok(ram_path, min_bytes=10_000_000):
        print('Downloading RAM (xinyu1205/recognize-anything, HF Space)...')
        downloaded = hf_hub_download(
            repo_id='xinyu1205/recognize-anything',
            repo_type='space',
            filename='ram_swin_large_14m.pth',
        )
        shutil.copy(downloaded, ram_path)
        print('Done:', ram_path)
    else:
        print('RAM is already present, skipping download:', ram_path)

    dape_path = 'preset/models/DAPE.pth'
    if not file_ok(dape_path, min_bytes=1_000_000):
        print('Downloading DAPE.pth (Guaishou74851/AdcSR mirror)...')
        downloaded = hf_hub_download(
            repo_id='Guaishou74851/AdcSR',
            filename='weight/pretrained/DAPE.pth',
        )
        shutil.copy(downloaded, dape_path)
        print('Done:', dape_path)
    else:
        print('DAPE is already present, skipping download:', dape_path)

    osediff_path = 'preset/models/osediff.pkl'
    if not file_ok(osediff_path, min_bytes=1_000_000):
        print('Downloading osediff.pkl (Guaishou74851/AdcSR mirror)...')
        downloaded = hf_hub_download(
            repo_id='Guaishou74851/AdcSR',
            filename='weight/pretrained/osediff.pkl',
        )
        shutil.copy(downloaded, osediff_path)
        print('Done:', osediff_path)
    else:
        print('osediff.pkl is already present, skipping download:', osediff_path)

    print('OSEDiff cloned, dependencies installed, weights are in place. Ready for the video pipeline.')

In [ ]:
#@title ##**Select Video File** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

upload_option = "Load from Google Drive Root"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive"]

file_name = None
last_selected_button = None

def reset_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

if upload_option == "Upload from PC":
    print("Please upload a video file.")
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
    else:
        print("No file uploaded.")
        file_name = None

elif upload_option == "Load from Google Drive Root":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for f in os.listdir(root_dir):
        if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in video_extensions:
            files_list.append(f)

    if not files_list:
        print("No video files found in Google Drive root.")
        file_name = None
    else:
        print("Select a video file from Google Drive root:")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

elif upload_option == "Load from Google Drive":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in video_extensions:
                relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                files_list.append(relative_path)

    if not files_list:
        print("No video files found in Google Drive or its subfolders.")
        file_name = None
    else:
        print("Select a video file from Google Drive (including subfolders):")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if file_name:
    print(f"Video file path set to: {file_name}")
else:
    print("Video file path not set. Please select a file.")

In [ ]:
#@title ##**Config Folders** { display-mode: "form" }
import os
import shutil
from google.colab import drive

output_folder = "google_drive" #@param ["google_drive","root"]

if output_folder == "google_drive":
    if not os.path.exists('/content/drive'):
        print("Google Drive не подключён. Подключаем...")
        drive.mount('/content/drive')
    real_output_folder = '/content/drive/MyDrive/osediff_output'
    real_input_folder = "/content/drive/MyDrive/osediff_input"
elif output_folder == "root":
    real_output_folder = '/content/osediff_output'
    real_input_folder = "/content/osediff_input"

if not os.path.exists(real_output_folder):
    os.makedirs(real_output_folder)

if not os.path.exists(real_input_folder):
    os.makedirs(real_input_folder)

#clear folders
clear_input_folder = False #@param {type:"boolean"}
up_to_frame = "" #@param {type:"string"}
from_frame = "" #@param {type:"string"}

def clean_folder(folder_path, up_to=None, from_frame=None):
    print(f"\nCurrent parameters:")
    print(f"Delete frames up to: {up_to if up_to else 'not specified'}")
    print(f"Delete frames after: {from_frame if from_frame else 'not specified'}")

    if not os.path.isdir(folder_path):
        print(f"\nFolder {folder_path} does not exist!")
        print("Creating a new folder...")
        os.makedirs(folder_path)
        return

    if not up_to and not from_frame:
        print("\nNo parameters specified - deleting all folder content...")
        shutil.rmtree(folder_path)
        os.makedirs(folder_path)
        print(f"Folder {folder_path} cleared and recreated")
        return

    print("\nStarting file processing...")
    files = os.listdir(folder_path)
    jpg_files = [f for f in files if f.endswith('.jpg')]

    if not jpg_files:
        print("No JPG files to process in the folder")
        return

    deleted_count = 0
    processed_count = 0

    for filename in jpg_files:
        try:
            frame_number = int(filename.split('.')[0])
            should_delete = False

            if up_to and from_frame:
                if frame_number < int(up_to) or frame_number > int(from_frame):
                    should_delete = True
            elif up_to:
                if frame_number < int(up_to):
                    should_delete = True
            elif from_frame:
                if frame_number > int(from_frame):
                    should_delete = True

            if should_delete:
                file_path = os.path.join(folder_path, filename)
                os.remove(file_path)
                deleted_count += 1
                if deleted_count <= 5:
                    print(f'File deleted: {filename}')
                elif deleted_count == 6:
                    print('...')
            else:
                processed_count += 1

        except ValueError:
            print(f'Skipped file with invalid name: {filename}')

    print(f'\nProcessing complete:')
    print(f'Total files: {len(jpg_files)}')
    print(f'Files deleted: {deleted_count}')
    print(f'Files retained: {processed_count}')

if clear_input_folder:
    up_to_frame = up_to_frame if up_to_frame != "0" else None
    from_frame = from_frame if from_frame != "0" else None
    clean_folder(real_input_folder, up_to_frame, from_frame)

clear_output_folder = False #@param {type:"boolean"}

if clear_output_folder:
    if os.path.isdir(real_output_folder):
        shutil.rmtree(real_output_folder)
    os.makedirs(real_output_folder)

In [ ]:
#@title ##**Run sequence (Extract Frames)** { display-mode: "form" }
import cv2
import imageio
import os
import tqdm
import subprocess
import numpy as np
import time

library = "ffmpeg" #@param ["cv2","imageio","ffmpeg","skvideo","scipy","moviepy"]
delay = "0.1" #@param [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

if (library == "ffmpeg"):
    import ffmpeg
    # Using file_name set from Cell 2
    full_path = file_name

    probe = ffmpeg.probe(full_path)
    video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    fps = video_info['r_frame_rate']
    duration = float(video_info['duration'])
    frame_count = int(video_info['nb_frames'])

    print("FPS: ", fps)
    print("Duration: ", duration)
    print("Frames: ", frame_count)

    pbar_ffmpeg = tqdm.tqdm(total=frame_count, ncols=100, position=0, leave=True)
    process = (
        ffmpeg
        .input(full_path)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24', qscale=0)
        .run_async(pipe_stdout=True)
    )

    for i in range(frame_count):
        try:
            raw_video = process.stdout.read(video_info['width'] * video_info['height'] * 3)
            frame = np.frombuffer(raw_video, dtype='uint8').reshape((video_info['height'], video_info['width'], 3))
            frame_path = f"{real_input_folder}/{i:09d}.jpg"
            if os.path.isfile(frame_path):
              pbar_ffmpeg.update(1)
              continue
            imageio.imwrite(frame_path, frame)
        except Exception as e:
            print(f"Error writing to disk: {str(e)}. Retrying...")
            continue
        pbar_ffmpeg.update(1)
        time.sleep(float(delay))

    pbar_ffmpeg.close()
    process.wait()

#check frames
def check_frames():
    frame_dir = real_input_folder
    frames = [int(f.split('.')[0].replace('frame', '')) for f in os.listdir(frame_dir) if f.lower().endswith(('.jpg', '.png'))]
    if not frames:
        print("No frames found.")
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print(min_frame)
    print(max_frame)

    missing_frames = []
    for i in range(min_frame, max_frame+1):
        if i not in frames:
            missing_frames.append(i)

    if len(missing_frames) > 0:
        print(f"Missing frames: {missing_frames}")
    else:
        print("All frames present")

attempts = 0
max_attempts = 10

while attempts < max_attempts:
    try:
        check_frames()
        break
    except Exception as e:
        attempts += 1
        print(f"Attempt {attempts} failed with error: {str(e)}")
        if attempts == max_attempts:
            print("Maximum attempts reached. Execution failed.")
        else:
            print("Retrying...")

In [ ]:
#@title ##**Run OSEDiff Upscale on Frames** { display-mode: "form" }
import os
import shutil
import subprocess
import re
import collections
import time
import torch
import gc
from PIL import Image
from tqdm.notebook import tqdm

# --- OSEDiff Parameters -------------------------------------------------------
task = "sr" #@param ["sr", "face"]
upscale = 3 #@param {type:"slider", min:1, max:8, step:1}
align_method = "adain" #@param ["adain", "wavelet", "nofix"]
mixed_precision = "fp16" #@param ["fp16", "fp32"]
merge_and_unload_lora = True #@param {type:"boolean"}

# Memory optimization (Tiling)
low_vram_tiled_sampling = True #@param {type:"boolean"}
vae_encoder_tiled_size = 512 #@param {type:"integer"}
vae_decoder_tiled_size = 128 #@param {type:"integer"}
latent_tiled_size = 96 #@param {type:"integer"}
latent_tiled_overlap = 16 #@param {type:"integer"}

batch_size = 1 #@param {type:"slider", min:1, max:10, step:1}
show_full_log = False #@param {type:"boolean"}

osediff_path = 'preset/models/osediff_face.pkl' if task == 'face' else 'preset/models/osediff.pkl'

# Универсальное извлечение номера кадра из любого имени
def extract_frame_number(filename):
    match = re.search(r'\d+', filename)
    return int(match.group()) if match else None

# Check frames input directory
def check_frames_output():
    frame_dir = real_input_folder
    frames = []
    for f in os.listdir(frame_dir):
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            num = extract_frame_number(f)
            if num is not None:
                frames.append(num)

    if not frames:
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print("Input range:", min_frame, "-", max_frame)

    missing_frames = []
    for i in range(min_frame, max_frame+1):
        if i not in frames:
            missing_frames.append(i)

    if len(missing_frames) > 0:
        print(f"Missing frames: {missing_frames}")
    else:
        print("All frames present")

check_frames_output()

if not os.path.exists(os.getcwd() + '/test_osediff.py') and os.path.isdir('/content/OSEDiff'):
    os.chdir('/content/OSEDiff')
if os.path.basename(os.getcwd()) != 'OSEDiff':
    raise RuntimeError("Folder /content/OSEDiff not found. Please run the 'Install' cell first.")

# --- Check required weights before running -----------------------------------
required_paths = {
    'SD21 Base': 'preset/models/stable-diffusion-2-1-base/model_index.json',
    'RAM': 'preset/models/ram_swin_large_14m.pth',
    'DAPE': 'preset/models/DAPE.pth',
    os.path.basename(osediff_path): osediff_path,
}
missing_weights = [name for name, p in required_paths.items() if not os.path.exists(p)]
if missing_weights:
    if task == 'face' and os.path.basename(osediff_path) in missing_weights:
        print(f"⚠️ {osediff_path} not found. Falling back to osediff.pkl (standard SR).")
        osediff_path = 'preset/models/osediff.pkl'
        missing_weights = [name for name, p in required_paths.items() if name != os.path.basename(osediff_path) and not os.path.exists(p)]
    if missing_weights:
        raise RuntimeError(f"Missing weights: {missing_weights}. Please re-run the 'Install' cell.")

if task == 'face':
    upscale = 1
    align_method = 'nofix'

# Folders for OSEDiff script batching (frames get copied here in batches, one image at a time)
upload_folder = "/content/OSEDiff/input_images"
result_folder = "/content/OSEDiff/output_images"

file_list = os.listdir(real_input_folder)
file_list.sort()
frames = []
for f in file_list:
    if f.lower().endswith(('.jpg', '.png', '.jpeg')):
        num = extract_frame_number(f)
        if num is not None:
            frames.append(num)
min_frame = min(frames) if frames else 0

real_files = os.listdir(real_output_folder)
if real_files:
    real_frames = []
    for f in real_files:
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            num = extract_frame_number(f)
            if num is not None:
                real_frames.append(num)
    start_frame = max(real_frames) + 1 if real_frames else min_frame
else:
    start_frame = min_frame

max_frame = frames[-1] if frames else 0
input_ext = None
for ext in ('.jpg', '.jpeg', '.png'):
    if file_list and any(f.endswith(ext) for f in file_list):
        input_ext = ext
        break
input_ext = input_ext or '.jpg'
files_to_copy = [f"{real_input_folder}/{frame:09d}{input_ext}" for frame in range(start_frame, max_frame+1) if f"{frame:09d}{input_ext}" in file_list]

total_files = len(files_to_copy)
print(f"Resuming from frame {start_frame}. Total frames left: {total_files}")

i = 0
retry_count = 0
max_retries = 3

# Оптимизация памяти PyTorch от фрагментации
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONUNBUFFERED"] = "1"

with tqdm(total=total_files, desc="Processing frames") as pbar:
    while i < total_files:
        batch_files = files_to_copy[i:i+batch_size]

        if os.path.isdir(upload_folder):
            shutil.rmtree(upload_folder)
        os.makedirs(upload_folder)
        if os.path.isdir(result_folder):
            shutil.rmtree(result_folder)
        os.makedirs(result_folder)

        # test_osediff.py only picks up files matching *.png (glob.glob), so every
        # frame gets (re-)encoded as a real PNG here, regardless of its source format.
        for file in batch_files:
            base = os.path.splitext(os.path.basename(file))[0]
            dst = os.path.join(upload_folder, base + '.png')
            with Image.open(file) as im:
                im.convert('RGB').save(dst, format='PNG')

        # Очистка памяти перед запуском
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        cmd = [
            "python", "-u", "test_osediff.py",
            "-i", upload_folder,
            "-o", result_folder,
            "--upscale", str(upscale),
            "--osediff_path", osediff_path,
            "--pretrained_model_name_or_path", "preset/models/stable-diffusion-2-1-base",
            "--ram_path", "preset/models/ram_swin_large_14m.pth",
            "--ram_ft_path", "preset/models/DAPE.pth",
            "--align_method", align_method,
            "--mixed_precision", mixed_precision,
        ]
        if merge_and_unload_lora:
            cmd += ["--merge_and_unload_lora", "True"]
        if low_vram_tiled_sampling:
            cmd += [
                "--vae_encoder_tiled_size", str(vae_encoder_tiled_size),
                "--vae_decoder_tiled_size", str(vae_decoder_tiled_size),
                "--latent_tiled_size", str(latent_tiled_size),
                "--latent_tiled_overlap", str(latent_tiled_overlap),
            ]

        process = None
        log_queue = collections.deque(maxlen=30)

        try:
            # Запуск скрипта
            process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

            for line in process.stdout:
                if show_full_log:
                    print(line, end="")
                log_queue.append(line)

            process.wait()

        except KeyboardInterrupt:
            print("\n[ОСТАНОВКА] Прервано пользователем. Очистка...")
            raise

        finally:
            if process is not None and process.poll() is None:
                process.terminate()
                process.wait()

        # Проверка результатов (test_osediff.py also creates a "txt" subfolder with
        # captions inside result_folder — only image files are counted/copied here)
        output_files = [
            f for f in (os.listdir(result_folder) if os.path.exists(result_folder) else [])
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]

        if process.returncode != 0 or len(output_files) != len(batch_files):
            retry_count += 1
            print(f"\n[ОШИБКА] Обработка прервалась. Попытка {retry_count} из {max_retries}.")

            if not show_full_log:
                print("--- ПОСЛЕДНИЕ СТРОКИ ЛОГА ---")
                for l in log_queue:
                    print(l, end="")
                print("-----------------------------")

            if retry_count >= max_retries:
                raise RuntimeError("Критическая ошибка (OOM). Уменьшите upscale или tile size.")

            time.sleep(2)
            continue # Повтор того же кадра

        # Если успех - записываем файлы на диск СРАЗУ с правильными чистыми именами
        retry_count = 0
        for out_file in output_files:
            frame_num = extract_frame_number(out_file)
            if frame_num is not None:
                ext = out_file.split('.')[-1]
                clean_name = f"{frame_num:09d}.{ext}"
                # Копируем файл из временной папки на диск с новым чистым именем
                shutil.copy(
                    os.path.join(result_folder, out_file),
                    os.path.join(real_output_folder, clean_name)
                )

        i += len(batch_files)
        pbar.update(len(batch_files))

# Функция финальной проверки кадров в папке вывода
def check_final_frames():
    frame_dir = real_output_folder
    frames = []
    for f in os.listdir(frame_dir):
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            num = extract_frame_number(f)
            if num is not None:
                frames.append(num)

    if not frames:
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print("\nOutput range:", min_frame, "-", max_frame)

    missing_frames = []
    for i in range(min_frame, max_frame+1):
        if i not in frames:
            missing_frames.append(i)

    if len(missing_frames) > 0:
        print(f"Missing output frames: {missing_frames}")
    else:
        print("All output frames present")

check_final_frames()


In [ ]:
#@title ##**Create video** { display-mode: "form" }
import cv2
import os
import subprocess
import time
import numpy as np
from tqdm.notebook import tqdm
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

def log_time(start, message):
    elapsed = time.time() - start
    print(f"{message}: {elapsed:.2f} seconds")
    return time.time()

start_time = time.time()

upscaled_image = 70 #@param {type:"slider", min:0, max:100, step:1}
sharpness = 3 #@param {type:"slider", min:0.0, max:7.0, step:0.1}

# Тот же алгоритм резкости, что и в ячейке "Visualization": кадр сглаживается
# ядром 3x3 (сумма весов 13, центр 5), затем результат экстраполируется между
# сглаженным и оригинальным кадром: out = smoothed*(1-factor) + frame*factor.
# factor=1.0 - кадр без изменений (по умолчанию), <1.0 - смягчение, >1.0 - резкость выше исходной.
_sharpen_kernel = np.array([[1, 1, 1],
                            [1, 5, 1],
                            [1, 1, 1]], dtype=np.float32) / 13.0

def apply_sharpness(frame, factor):
    if factor == 1.0:
        return frame
    smoothed = cv2.filter2D(frame, -1, _sharpen_kernel, borderType=cv2.BORDER_REPLICATE)
    result = smoothed.astype(np.float32) * (1 - factor) + frame.astype(np.float32) * factor
    return np.clip(result, 0, 255).astype(np.uint8)

print(f"output_folder: {output_folder}")

if 'file_name' in locals() and os.path.exists(file_name):
    base_file_name = os.path.basename(file_name)
else:
    raise ValueError("file_name is not defined or the file does not exist")

if output_folder == "google_drive":
    save_path = '/content/drive/MyDrive/'
elif output_folder == "root":
    save_path = '/content/'
else:
    save_path = '/content/'

full_path = os.path.join(save_path, base_file_name) if not os.path.exists(file_name) else file_name
output_file_name = base_file_name.rsplit('.', 1)[0] + f'_OSEDiff_upscale_{upscaled_image}.mp4'
output_file = os.path.join(save_path, output_file_name)
temp_video = "/content/temp_video.mp4"

start_time = log_time(start_time, "Initial setup")

cap = cv2.VideoCapture(full_path)
fps_of_video = int(cap.get(cv2.CAP_PROP_FPS))
cap.release()

upscaled_img_files = [os.path.join(real_output_folder, img) for img in os.listdir(real_output_folder) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
upscaled_img_files.sort()

if upscaled_image < 100:
    original_img_files = [os.path.join(real_input_folder, img) for img in os.listdir(real_input_folder) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
    original_img_files.sort()
    if len(upscaled_img_files) != len(original_img_files):
        raise ValueError("Number of upscaled and original frames does not match")

if upscaled_img_files:
    first_frame = cv2.imread(upscaled_img_files[0], cv2.IMREAD_COLOR)
    height, width = first_frame.shape[:2]

    needs_resize = False
    for img in upscaled_img_files[:10]:
        frame = cv2.imread(img, cv2.IMREAD_COLOR)
        if frame.shape[:2] != (height, width):
            needs_resize = True
            break
        del frame
    del first_frame
else:
    raise ValueError("No images found in the upscaled frames folder")

start_time = log_time(start_time, "Frame list preparation")

def get_video_bitrate(file_path):
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', file_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    bitrate = result.stdout.strip()
    try:
        return int(bitrate)
    except ValueError:
        return None

bitrate = get_video_bitrate(full_path)
if bitrate:
    bitrate = int(bitrate * 1.5)
    bitrate_str = f'{bitrate // 1000}k'
else:
    bitrate_str = '7500k'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_video, fourcc, fps_of_video, (width, height))

if upscaled_image == 100:
    for img_file in tqdm(upscaled_img_files, desc="Processing frames"):
        frame = cv2.imread(img_file, cv2.IMREAD_COLOR)
        if needs_resize and frame.shape[:2] != (height, width):
            frame = cv2.resize(frame, (width, height))
        frame = apply_sharpness(frame, sharpness)
        out.write(frame)
        del frame
else:
    alpha = upscaled_image / 100.0
    beta = 1 - alpha
    for upscaled_img, original_img in tqdm(zip(upscaled_img_files, original_img_files), total=len(upscaled_img_files), desc="Processing frames"):
        upscaled_frame = cv2.imread(upscaled_img, cv2.IMREAD_COLOR)
        original_frame = cv2.imread(original_img, cv2.IMREAD_COLOR)

        if needs_resize and upscaled_frame.shape[:2] != (height, width):
            upscaled_frame = cv2.resize(upscaled_frame, (width, height))

        upscaled_frame = apply_sharpness(upscaled_frame, sharpness)

        original_frame_resized = cv2.resize(original_frame, (width, height))

        blended_frame = cv2.addWeighted(upscaled_frame, alpha, original_frame_resized, beta, 0)

        out.write(blended_frame)
        del upscaled_frame, original_frame, original_frame_resized, blended_frame

out.release()
gc.collect()

start_time = log_time(start_time, "Frame processing and writing")

temp_converted = "/content/temp_converted.mp4"
cmd = ['ffmpeg', '-i', temp_video, '-c:v', 'libx264', '-b:v', bitrate_str, '-y', temp_converted]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg conversion failed: {result.stderr}")
    raise RuntimeError("Conversion to libx264 failed")
os.remove(temp_video)
os.rename(temp_converted, temp_video)

start_time = log_time(start_time, "FFmpeg conversion to libx264")

cmd = ['ffmpeg', '-i', temp_video, '-i', full_path, '-map', '0:v', '-map', '1:a?', '-map', '1:s?', '-c', 'copy', '-y', output_file]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg audio muxing failed: {result.stderr}")
    raise RuntimeError("Audio muxing failed")

start_time = log_time(start_time, "Final audio and subtitles muxing")

if os.path.exists(output_file):
    if os.path.exists(temp_video):
        os.remove(temp_video)
    print("Video created successfully")
    print(f"Video saved at: {output_file}")
else:
    print("Failed to create video")
    print(f"Expected save path: {output_file}")
    print(f"FFmpeg error output: {result.stderr}")

start_time = log_time(start_time, "Cleanup")